# Convert SROIE Benchmark Results to Comparison Notebook Format

The [SROIE benchmark notebook](https://github.com/tmnestor/SROIE) outputs:
- **Per-image CSV**: `sroie_internvl3_per_image.csv` — columns: `image_id, {field}_gt, {field}_pred, {field}_match`
- **Summary JSON**: `sroie_internvl3_summary.json` — model name, elapsed time, per-field metrics

The comparison notebook (`model_comparison.ipynb`) expects:
- `evaluation_data/sroie_ground_truth.yaml`
- `results/sroie_{model_key}_results.yaml`

This notebook converts between the two formats.

---
## 1. Configuration

Set paths to the SROIE benchmark output files and the desired model key.

In [ ]:
from pathlib import Path

# --- Input: paths to SROIE benchmark output ---
SROIE_CSV = Path("path/to/sroie_internvl3_per_image.csv")
SROIE_SUMMARY = Path("path/to/sroie_internvl3_summary.json")

# --- Model key (used in output filenames) ---
# Must match the key in comparison_config.yaml under 'models:'
MODEL_KEY = "internvl3_5_8b"

# --- Output: base directory for the comparison notebook ---
OUTPUT_DIR = Path(".")  # Write files relative to this notebook's directory

# --- SROIE fields ---
SROIE_FIELDS = ("company", "date", "address", "total")

---
## 2. Load SROIE Benchmark Output

In [ ]:
import csv
import json

import yaml

# Load per-image CSV
assert SROIE_CSV.exists(), f"CSV not found: {SROIE_CSV.resolve()}"
with SROIE_CSV.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# Load summary JSON
assert SROIE_SUMMARY.exists(), f"Summary not found: {SROIE_SUMMARY.resolve()}"
summary = json.loads(SROIE_SUMMARY.read_text(encoding="utf-8"))

print(f"Loaded {len(rows)} image results")
print(f"Model: {summary.get('model', 'unknown')}")
print(f"Elapsed: {summary.get('elapsed_seconds', 0):.1f}s")
print(f"Overall F1 (from benchmark): {summary.get('overall_f1', 0):.4f}")
print(f"\nCSV columns: {list(rows[0].keys()) if rows else '(empty)'}")

---
## 3. Build Ground Truth YAML

Extract `{field}_gt` columns from the CSV into the ground truth structure.

In [ ]:
gt_images: dict[str, dict[str, str]] = {}

for row in rows:
    image_id = row["image_id"]
    fields = {}
    for field in SROIE_FIELDS:
        value = row.get(f"{field}_gt", "")
        if value:
            fields[field] = value
    gt_images[image_id] = fields

ground_truth = {"images": gt_images}

# Preview
sample_id = next(iter(gt_images))
print(f"Ground truth: {len(gt_images)} images")
print(f"\nSample ({sample_id}):")
for k, v in gt_images[sample_id].items():
    print(f"  {k}: {v}")

---
## 4. Build Model Results YAML

Extract `{field}_pred` columns from the CSV. Per-image `processing_time` is
estimated as `elapsed_seconds / total_images` (the SROIE benchmark only records
total elapsed time, not per-image timing).

In [ ]:
elapsed = summary.get("elapsed_seconds", 0.0)
total_images = summary.get("total_images", len(rows))
per_image_time = round(elapsed / total_images, 3) if total_images > 0 else 0.0

results_list = []
for row in rows:
    fields = {}
    for field in SROIE_FIELDS:
        value = row.get(f"{field}_pred", "")
        if value:
            fields[field] = value
    results_list.append(
        {
            "image": row["image_id"],
            "processing_time": per_image_time,
            "fields": fields,
        }
    )

model_results = {"results": results_list}

print(f"Results: {len(results_list)} images")
print(f"Estimated per-image time: {per_image_time:.3f}s")
print(f"\nSample ({results_list[0]['image']}):")
for k, v in results_list[0]["fields"].items():
    print(f"  {k}: {v}")

---
## 5. Write YAML Files

In [ ]:
# Ground truth
gt_path = OUTPUT_DIR / "evaluation_data" / "sroie_ground_truth.yaml"
gt_path.parent.mkdir(parents=True, exist_ok=True)
gt_path.write_text(
    yaml.dump(
        ground_truth, default_flow_style=False, allow_unicode=True, sort_keys=False
    ),
    encoding="utf-8",
)
print(f"Written: {gt_path} ({len(gt_images)} images)")

# Model results
results_path = OUTPUT_DIR / "results" / f"sroie_{MODEL_KEY}_results.yaml"
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(
    yaml.dump(
        model_results, default_flow_style=False, allow_unicode=True, sort_keys=False
    ),
    encoding="utf-8",
)
print(f"Written: {results_path} ({len(results_list)} images)")

---
## 6. Verify

Quick sanity check: reload the YAML files and confirm they parse correctly.

In [ ]:
# Reload and validate
gt_check = yaml.safe_load(gt_path.read_text(encoding="utf-8"))
results_check = yaml.safe_load(results_path.read_text(encoding="utf-8"))

assert "images" in gt_check, "Ground truth missing 'images' key"
assert "results" in results_check, "Results missing 'results' key"
assert len(gt_check["images"]) == len(rows), "Image count mismatch in ground truth"
assert len(results_check["results"]) == len(rows), "Image count mismatch in results"

# Check a sample entry has the expected structure
sample_result = results_check["results"][0]
assert "image" in sample_result, "Result entry missing 'image' key"
assert "processing_time" in sample_result, "Result entry missing 'processing_time' key"
assert "fields" in sample_result, "Result entry missing 'fields' key"

print("Validation passed.")
print(f"\nFiles ready for model_comparison.ipynb:")
print(f"  {gt_path}")
print(f"  {results_path}")